# A/B Test 电商页面转化率分析

**项目背景**：电商平台对产品详情页进行改版，验证新版页面（new_page）是否比旧版（old_page）带来更高的购买转化率。

**数据集**：来自 Kaggle，约 29 万条用户访问记录，包含分组信息与是否转化。

**分析流程**：
```
Step 0 → 数据加载与初步了解
Step 1 → 数据清洗（处理分组错位与重复用户）
Step 2 → 探索性分析 EDA（转化率对比 + 趋势图）
Step 3 → 假设检验（双尾 z 检验）→ 最终结论
```

---

## 实验假设设定

| 项目 | 内容 |
|------|------|
| 原假设 H₀ | p_new = p_old（新旧页面转化率无显著差异） |
| 备择假设 H₁ | p_new ≠ p_old（新旧页面转化率存在显著差异） |
| 检验方式 | 双尾 z 检验 |
| 显著性水平 | α = 0.05 |
| 统计效能目标 | 1 - β = 0.80 |

> 采用双尾检验，因为事先不确定新页面一定优于旧页面，也存在更差的可能性。

In [ ]:
# 导入所需库
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest, proportion_confint
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

print('✅ 库导入完成')

---
## Step 0 · 数据加载与初步了解

In [ ]:
df = pd.read_csv('ab_data.csv')

print(f'数据总行数：{len(df):,}')
print(f'字段列表：{list(df.columns)}')
df.head()

In [ ]:
print('【各字段取值分布】')
print('\ngroup 分组：')
print(df['group'].value_counts())
print('\nlanding_page 页面：')
print(df['landing_page'].value_counts())
print('\nconverted 转化（0=未转化，1=转化）：')
print(df['converted'].value_counts())

In [ ]:
# 样本量评估 —— Power Analysis
baseline_rate = df[df['group'] == 'control']['converted'].mean()
expected_lift = 0.02
new_rate = baseline_rate + expected_lift

effect_size = proportion_effectsize(new_rate, baseline_rate)
analysis = NormalIndPower()
required_n = analysis.solve_power(effect_size=effect_size, alpha=0.05,
                                   power=0.80, alternative='two-sided')

print(f'基准转化率（对照组）：{baseline_rate*100:.2f}%')
print(f'期望检测的最小提升：+{expected_lift*100:.1f}%')
print(f'每组所需最少样本量：{int(np.ceil(required_n)):,}')
print(f'实际对照组样本量：  {len(df[df["group"]=="control"]):,}')
print(f'实际实验组样本量：  {len(df[df["group"]=="treatment"]):,}')
print('\n✅ 实际样本量远超要求，统计效能充足')

---
## Step 1 · 数据清洗

**发现的异常**：`group` 字段两组人数不对等（147,276 vs 147,202），但 `landing_page` 字段 new_page 和 old_page 数量却完全相等（各 147,239）。说明部分用户的分组与页面存在错位分配，必须清除。

In [ ]:
# 检查不匹配记录
mismatch = df[
    ((df['group'] == 'control')   & (df['landing_page'] == 'new_page')) |
    ((df['group'] == 'treatment') & (df['landing_page'] == 'old_page'))
]
print(f'分组与页面不匹配的记录数：{len(mismatch):,}')
mismatch.head(3)

In [ ]:
# 执行清洗
df_clean = df[
    ((df['group'] == 'control')   & (df['landing_page'] == 'old_page')) |
    ((df['group'] == 'treatment') & (df['landing_page'] == 'new_page'))
].copy()

df_clean['timestamp'] = pd.to_datetime(df_clean['timestamp'])
df_clean = df_clean.sort_values('timestamp').drop_duplicates(subset='user_id', keep='first')

# 清洗前后对比
summary = pd.DataFrame({
    '指标': ['总行数', '对照组样本量', '实验组样本量', '对照组转化率', '实验组转化率'],
    '清洗前': [
        f"{len(df):,}",
        f"{len(df[df['group']=='control']):,}",
        f"{len(df[df['group']=='treatment']):,}",
        f"{df[df['group']=='control']['converted'].mean()*100:.2f}%",
        f"{df[df['group']=='treatment']['converted'].mean()*100:.2f}%"
    ],
    '清洗后': [
        f"{len(df_clean):,}",
        f"{len(df_clean[df_clean['group']=='control']):,}",
        f"{len(df_clean[df_clean['group']=='treatment']):,}",
        f"{df_clean[df_clean['group']=='control']['converted'].mean()*100:.2f}%",
        f"{df_clean[df_clean['group']=='treatment']['converted'].mean()*100:.2f}%"
    ]
})
summary

---
## Step 2 · 探索性分析（EDA）

In [ ]:
control   = df_clean[df_clean['group'] == 'control']
treatment = df_clean[df_clean['group'] == 'treatment']

rate_ctrl = control['converted'].mean()
rate_trt  = treatment['converted'].mean()
se_ctrl   = np.sqrt(rate_ctrl * (1 - rate_ctrl) / len(control))
se_trt    = np.sqrt(rate_trt  * (1 - rate_trt)  / len(treatment))

diff = rate_trt - rate_ctrl
se_diff = np.sqrt(se_ctrl**2 + se_trt**2)

print(f'对照组转化率：{rate_ctrl*100:.4f}%  95% CI：[{(rate_ctrl-1.96*se_ctrl)*100:.4f}%, {(rate_ctrl+1.96*se_ctrl)*100:.4f}%]')
print(f'实验组转化率：{rate_trt*100:.4f}%  95% CI：[{(rate_trt-1.96*se_trt)*100:.4f}%, {(rate_trt+1.96*se_trt)*100:.4f}%]')
print(f'差值：{diff*100:.4f}%  差值 95% CI：[{(diff-1.96*se_diff)*100:.4f}%, {(diff+1.96*se_diff)*100:.4f}%]')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('A/B Test 探索性分析', fontsize=14, fontweight='bold')

# 柱状图
labels = ['对照组\n(old_page)', '实验组\n(new_page)']
rates  = [rate_ctrl, rate_trt]
errors = [1.96 * se_ctrl, 1.96 * se_trt]
colors = ['#4C72B0', '#DD8452']
bars = axes[0].bar(labels, [r*100 for r in rates], color=colors, width=0.45,
                   yerr=[e*100 for e in errors], capsize=6,
                   error_kw={'linewidth': 1.5})
axes[0].set_title('两组转化率对比（含 95% CI）', fontsize=12)
axes[0].set_ylabel('转化率 (%)')
axes[0].set_ylim(0, max([r*100 for r in rates]) * 1.3)
for bar, rate in zip(bars, rates):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.15,
                 f'{rate*100:.2f}%', ha='center', va='bottom', fontsize=11)
axes[0].grid(axis='y', linestyle='--', alpha=0.4)
axes[0].spines[['top','right']].set_visible(False)

# 每日趋势图
df_clean['date'] = df_clean['timestamp'].dt.date
daily = df_clean.groupby(['date','group']).agg(n=('converted','count'), conv=('converted','sum')).reset_index()
daily['rate'] = daily['conv'] / daily['n']
daily['date'] = pd.to_datetime(daily['date'])
for grp, color, label in [('control','#4C72B0','对照组（old_page）'),('treatment','#DD8452','实验组（new_page）')]:
    d = daily[daily['group']==grp].sort_values('date')
    axes[1].plot(d['date'], d['rate']*100, color=color, label=label, linewidth=1.8, marker='o', markersize=3)
axes[1].set_title('每日转化率趋势', fontsize=12)
axes[1].set_ylabel('转化率 (%)')
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
axes[1].xaxis.set_major_locator(mdates.WeekdayLocator(interval=1))
plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30, ha='right')
axes[1].legend(fontsize=10)
axes[1].grid(linestyle='--', alpha=0.4)
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()

**EDA 小结**：两组转化率极为接近（12.04% vs 11.88%），置信区间大幅重叠，每日趋势图两条线全程交织，无持续性分叉，说明实验期间无外部干扰。差值置信区间包含 0，初步判断差异不显著，将由假设检验正式验证。

---
## Step 3 · 假设检验（双尾 z 检验）

In [ ]:
n_ctrl    = len(control)
n_trt     = len(treatment)
conv_ctrl = control['converted'].sum()
conv_trt  = treatment['converted'].sum()

z_stat, p_value = proportions_ztest(
    np.array([conv_trt, conv_ctrl]),
    np.array([n_trt, n_ctrl]),
    alternative='two-sided'
)

print(f'z 统计量：{z_stat:.4f}')
print(f'p 值：    {p_value:.4f}')
print()
if p_value < 0.05:
    print('✅ p < 0.05，拒绝 H₀ → 两组转化率存在统计显著差异')
else:
    print(f'❌ p = {p_value:.4f} ≥ 0.05，未能拒绝 H₀ → 两组转化率无统计显著差异')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('A/B Test 假设检验结果', fontsize=14, fontweight='bold')

# z 分布曲线
x = np.linspace(-5, 5, 1000)
y = (1 / np.sqrt(2 * np.pi)) * np.exp(-0.5 * x**2)
z_crit = 1.96
axes[0].plot(x, y, color='#333333', linewidth=2)
axes[0].fill_between(x, y, where=(x <= -z_crit), color='#E84B4B', alpha=0.4, label='拒绝域 (α/2=0.025)')
axes[0].fill_between(x, y, where=(x >= z_crit),  color='#E84B4B', alpha=0.4)
axes[0].fill_between(x, y, where=((x > -z_crit) & (x < z_crit)), color='#A8D5A2', alpha=0.3, label='接受域')
axes[0].axvline(z_stat, color='#1A6FB5', linewidth=2.5, linestyle='--', label=f'z = {z_stat:.4f}')
axes[0].axvline(-z_crit, color='#E84B4B', linewidth=1.5, linestyle=':')
axes[0].axvline( z_crit, color='#E84B4B', linewidth=1.5, linestyle=':', label=f'临界值 ±{z_crit}')
axes[0].text(z_stat + 0.1, 0.3, f'p = {p_value:.4f}', color='#1A6FB5', fontsize=10)
axes[0].set_title('z 检验：标准正态分布与拒绝域', fontsize=11)
axes[0].set_xlabel('z 值')
axes[0].set_ylabel('概率密度')
axes[0].legend(fontsize=9)
axes[0].grid(linestyle='--', alpha=0.3)
axes[0].spines[['top','right']].set_visible(False)

# 置信区间点图
ci_ctrl = proportion_confint(conv_ctrl, n_ctrl, alpha=0.05, method='normal')
ci_trt  = proportion_confint(conv_trt,  n_trt,  alpha=0.05, method='normal')
for i, (label, rate, ci, color) in enumerate([
    ('对照组\n(old_page)', rate_ctrl, ci_ctrl, '#4C72B0'),
    ('实验组\n(new_page)', rate_trt,  ci_trt,  '#DD8452')
]):
    axes[1].errorbar(i, rate*100, yerr=(rate-ci[0])*100,
                     fmt='o', color=color, markersize=10,
                     capsize=8, capthick=2, linewidth=2)
    axes[1].text(i+0.07, rate*100, f'{rate*100:.2f}%', va='center', fontsize=10, color=color)
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(['对照组\n(old_page)', '实验组\n(new_page)'], fontsize=10)
axes[1].set_xlim(-0.5, 1.5)
axes[1].set_ylim(11.5, 12.5)
axes[1].set_title('两组转化率与 95% 置信区间', fontsize=11)
axes[1].set_ylabel('转化率 (%)')
axes[1].grid(axis='y', linestyle='--', alpha=0.3)
axes[1].spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.show()

---
## 最终结论

| 判据 | 结果 | 解读 |
|------|------|------|
| z 统计量 | -1.3109 | 落在接受域（-1.96, +1.96）内 |
| p 值 | 0.1899 | ≥ 0.05，约 19% 概率此差异为随机波动 |
| 差值 95% CI | [-0.39%, +0.08%] | 包含 0，无法排除两组无差异的可能 |

**三项判据一致：未能拒绝原假设 H₀。**

> 新页面（new_page）未带来统计显著的转化率提升。观察到的 -0.16% 差异属于随机波动范围，不具备实际业务意义。

### 业务建议

不建议将新页面全量上线，可考虑以下方向：
1. 延长实验周期或扩大样本量，进一步确认结果
2. 针对特定用户群体（如新用户 / 老用户）做分层分析
3. 重新审视新页面的设计改动点，评估是否值得继续迭代